<a href="https://colab.research.google.com/github/tphan356/chicago-rideshare-pricing/blob/main/code/model/fare-by-hour-modeling/05_deep_learning_fare_prediction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Fare Prediction in Google Colab

## Goal

Notebook 04 showed that traditional models already perform well:

- Baseline Linear Regression: about R² 0.72–0.73
- Full Linear Regression: about R² 0.76
- Polynomial Ridge: about R² 0.77
- Gradient Boosting: about R² 0.78

The next question is:

> **Can deep learning learn additional nonlinear relationships from the large Chicago rideshare dataset?**

This notebook is designed for **Google Colab with GPU**.

It uses a tabular neural network with:

- standardized continuous variables
- learned embeddings for categorical variables
- dense neural-network layers
- optional hyperparameter tuning with Keras Tuner

The first experiment uses **500,000 trips** so the result is comparable with Notebook 04. After the model works correctly, increase the sample size to test whether deep learning benefits from more data.


## Modeling Flow

```text
Mount Google Drive
        ↓
Locate the cleaned Parquet file
        ↓
Copy it to Colab local disk
        ↓
Build the same fare/demand features
        ↓
500k reproducible sample
        ↓
80/20 train-test split
        ↓
Scale continuous features
        ↓
Embed categorical features
        ↓
Train neural network
        ↓
R² / MAE / RMSE
        ↓
Optional Keras Tuner
        ↓
Retrain best architecture
        ↓
Compare with Notebook 04
```


# Part I — Google Colab Setup

## Before Running

In Colab:

1. Open **Runtime**
2. Select **Change runtime type**
3. Choose a **GPU**
4. Save


In [ ]:
import sys

print("Python:", sys.version)
print("Running in Colab:", "google.colab" in sys.modules)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Running in Colab: True


## Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Locate `rideshare_2025_clean.parquet`

The notebook first checks:

```text
MyDrive/chicago-rideshare-pricing/data/processed/rideshare_2025_clean.parquet
```

In [ ]:
from pathlib import Path

DRIVE_FILE = Path(
    "/content/drive/MyDrive/chicago-rideshare-pricing/data/processed/rideshare_2025_clean.parquet"
)

if not DRIVE_FILE.exists():
    print("Default path not found. Searching MyDrive...")

    matches = list(
        Path("/content/drive/MyDrive").rglob(
            "rideshare_2025_clean.parquet"
        )
    )

    if len(matches) == 1:
        DRIVE_FILE = matches[0]

    elif len(matches) > 1:
        print("Multiple matches found:")
        for match in matches:
            print(match)

        raise FileNotFoundError(
            "Set DRIVE_FILE manually to the correct file."
        )

    else:
        raise FileNotFoundError(
            "rideshare_2025_clean.parquet was not found."
        )

print("Using:", DRIVE_FILE)
print(f"File size: {DRIVE_FILE.stat().st_size / 1024**3:.2f} GB")


Using: /content/drive/MyDrive/chicago-rideshare-pricing/data/processed/rideshare_2025_clean.parquet
File size: 1.81 GB


## Copy the Parquet File to Colab Local Disk

Reading repeatedly from mounted Google Drive is slower than reading from Colab's local disk.

The original stays in Drive; this creates only a temporary Colab copy.


In [ ]:
import shutil

LOCAL_FILE = Path(
    "/content/drive/MyDrive/chicago-rideshare-pricing/data/processed/rideshare_2025_clean.parquet"
)

if not LOCAL_FILE.exists():
    print("Copying file to Colab local disk...")
    shutil.copy2(DRIVE_FILE, LOCAL_FILE)
    print("Copy complete.")
else:
    print("Local copy already exists.")

print(
    f"Local size: "
    f"{LOCAL_FILE.stat().st_size / 1024**3:.2f} GB"
)


Local copy already exists.
Local size: 1.81 GB


## Install Extra Packages

TensorFlow is normally already installed in Colab.

This installs only the packages needed for data querying and optional tuning.


In [ ]:
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "duckdb",
    "pyarrow",
    "keras-tuner"
])

print("Packages installed.")


Packages installed.


## Import Libraries and Check the GPU


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print(
        "WARNING: No GPU detected. "
        "Use Runtime > Change runtime type > GPU."
    )


TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Main Settings

Start with 500,000 rows for a fair comparison with Notebook 04.


In [ ]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

SAMPLE_SIZE = 500_000
TEST_SIZE = 0.20
BATCH_SIZE = 2048
MAX_EPOCHS = 100

REPORTS_DIR = Path(
    "/content/drive/MyDrive/chicago-rideshare-pricing/reports"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Sample size: {SAMPLE_SIZE:,}")
print(f"Batch size: {BATCH_SIZE:,}")


Sample size: 500,000
Batch size: 2,048


# Part II — Build the Modeling Dataset

## Read the Parquet File with DuckDB

DuckDB reads directly from the local Parquet file, so the full 1.9 GB file is not loaded into pandas memory.


In [ ]:
con = duckdb.connect()

con.execute(f'''
    CREATE OR REPLACE VIEW rideshare AS

    SELECT *
    FROM read_parquet(
        '{LOCAL_FILE.as_posix()}'
    )
''')

con.sql('''
    SELECT
        COUNT(*) AS total_rows,
        MIN(trip_start_timestamp) AS first_trip,
        MAX(trip_start_timestamp) AS last_trip

    FROM rideshare
''').show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬─────────────────────┐
│ total_rows │     first_trip      │      last_trip      │
│   int64    │      timestamp      │      timestamp      │
├────────────┼─────────────────────┼─────────────────────┤
│   55075632 │ 2025-01-01 00:00:00 │ 2025-12-31 23:45:00 │
└────────────┴─────────────────────┴─────────────────────┘



## Basic Trip and Time Features

I keep the feature definitions consistent with Notebook 04.


In [ ]:
con.execute('''
    CREATE OR REPLACE VIEW rideshare_features AS

    SELECT
        CAST(fare AS DOUBLE) AS fare,
        trip_miles,
        trip_seconds / 60.0 AS trip_minutes,

        pickup_community_area,
        dropoff_community_area,

        CAST(trip_start_timestamp AS DATE) AS trip_date,
        CAST(EXTRACT(HOUR FROM trip_start_timestamp) AS INTEGER) AS hour,
        CAST(EXTRACT(DOW FROM trip_start_timestamp) AS INTEGER) AS day_of_week,
        CAST(EXTRACT(MONTH FROM trip_start_timestamp) AS INTEGER) AS month

    FROM rideshare
''')


## Local Demand

`local_demand` is the number of rides starting in the same pickup community, date, and hour.

It is calculated from the **full dataset before sampling**.


In [ ]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE hourly_demand AS

    SELECT
        trip_date,
        day_of_week,
        hour,
        pickup_community_area,
        COUNT(*) AS local_demand

    FROM rideshare_features

    GROUP BY
        trip_date,
        day_of_week,
        hour,
        pickup_community_area
''')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

##  Typical Demand and Demand Ratio

`demand_ratio` measures how busy an area is compared with its normal demand for the same weekday and hour.

```text
1.0 = normal demand
1.5 = 50% above normal
2.0 = twice normal
0.5 = half normal
```


In [ ]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE typical_demand AS

    SELECT
        pickup_community_area,
        day_of_week,
        hour,
        AVG(local_demand) AS typical_demand

    FROM hourly_demand

    GROUP BY
        pickup_community_area,
        day_of_week,
        hour
''')


##  Add Holiday Status and Combine Features


In [ ]:
holidays_2025 = [
    "2025-01-01", "2025-01-20", "2025-02-17",
    "2025-05-26", "2025-06-19", "2025-07-04",
    "2025-09-01", "2025-10-13", "2025-11-11",
    "2025-11-27", "2025-12-25"
]

holiday_sql = ", ".join(
    f"DATE '{date}'"
    for date in holidays_2025
)

con.execute(f'''
    CREATE OR REPLACE VIEW modeling_data AS

    SELECT
        r.fare,
        r.trip_miles,
        r.trip_minutes,

        d.local_demand,
        t.typical_demand,

        d.local_demand
        / NULLIF(t.typical_demand, 0)
            AS demand_ratio,

        CASE
            WHEN r.trip_minutes > 0
            THEN 60.0 * r.trip_miles / r.trip_minutes
            ELSE NULL
        END AS avg_speed_mph,

        r.hour,
        r.day_of_week,
        r.month,
        r.pickup_community_area,
        r.dropoff_community_area,

        CASE
            WHEN r.trip_date IN ({holiday_sql})
            THEN 1
            ELSE 0
        END AS is_holiday

    FROM rideshare_features AS r

    LEFT JOIN hourly_demand AS d
        ON r.trip_date = d.trip_date
        AND r.hour = d.hour
        AND r.pickup_community_area = d.pickup_community_area

    LEFT JOIN typical_demand AS t
        ON r.pickup_community_area = t.pickup_community_area
        AND r.day_of_week = t.day_of_week
        AND r.hour = t.hour
''')


# Part III — Create the Deep-Learning Sample

## Start with 500,000 Trips

This first test uses the same approximate sample size as Notebook 04.

No fare-based outlier removal is used.


In [ ]:
model_df = con.sql(f'''
    SELECT *
    FROM modeling_data

    WHERE fare > 0
      AND trip_miles > 0
      AND trip_minutes > 0
      AND local_demand IS NOT NULL
      AND typical_demand IS NOT NULL
      AND demand_ratio IS NOT NULL
      AND avg_speed_mph IS NOT NULL

    USING SAMPLE reservoir(
        {SAMPLE_SIZE} ROWS
    )

    REPEATABLE ({RANDOM_STATE})
''').df()

print(f"Modeling rows: {len(model_df):,}")

display(
    model_df.head()
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 497,948


,fare,trip_miles,trip_minutes,local_demand,typical_demand,demand_ratio,avg_speed_mph,hour,day_of_week,month,pickup_community_area,dropoff_community_area,is_holiday
0,5.0,1.6,6.566667,1157,1133.943396,1.020333,14.619289,20,3,1,28,33,0
1,15.0,2.9,14.566667,55,158.942308,0.346038,11.945080,21,6,11,31,28,0
2,7.5,1.7,8.716667,72,52.230769,1.378498,11.701721,14,2,3,23,27,0
3,20.0,5.2,17.200000,193,162.076923,1.190793,18.139535,8,4,1,3,8,0
4,20.0,8.1,40.116667,78,94.730769,0.823386,12.114666,12,5,5,25,8,0


## Inspect the Continuous Variables


In [ ]:
display(
    model_df[
        [
            "fare",
            "trip_miles",
            "trip_minutes",
            "local_demand",
            "demand_ratio",
            "avg_speed_mph"
        ]
    ].describe().T
)


,count,mean,std,min,25%,50%,75%,max
fare,497948.0,16.563582,13.718656,2.500000,7.500000,12.500000,20.000000,417.500000
trip_miles,497948.0,5.811704,7.174841,0.100000,1.600000,3.200000,7.000000,297.400000
trip_minutes,497948.0,17.275158,12.952871,0.683333,8.483333,13.533333,22.016667,280.200000
local_demand,497948.0,799.113247,748.755507,1.000000,163.000000,633.000000,1217.000000,4766.000000
demand_ratio,497948.0,1.088551,0.750757,0.027159,0.922923,1.030380,1.171698,43.561093
avg_speed_mph,497948.0,17.159222,9.397676,0.229299,10.746269,14.187192,20.970874,78.637275


## Create the 80/20 Train-Test Split

The test set remains untouched during both normal training and hyperparameter tuning.


In [ ]:
train_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")


Training rows: 398,358
Testing rows:  99,590


# Part IV — Prepare Neural-Network Inputs

## Standardize Continuous Variables

Neural networks generally train more easily when continuous variables are on similar scales.

The scaler is fitted only on the training data.


In [ ]:
numeric_features = [
    "trip_miles",
    "trip_minutes",
    "local_demand",
    "demand_ratio",
    "avg_speed_mph"
]

scaler = StandardScaler()

train_numeric = scaler.fit_transform(
    train_df[numeric_features]
).astype("float32")

test_numeric = scaler.transform(
    test_df[numeric_features]
).astype("float32")


##  Convert Categorical Variables to Integer IDs

The embedding layers use:

- hour: 0–23
- weekday: 0–6
- month: 0–11
- pickup area: 0–76
- drop-off area: 0–77
- holiday: 0–1


In [ ]:
def encode_categories(df):
    encoded = pd.DataFrame(index=df.index)
    encoded["hour"] = (df["hour"].astype(int))
    encoded["day_of_week"] = (df["day_of_week"].astype(int))
    encoded["month"] = (df["month"].astype(int) - 1)
    encoded["pickup_area"] = (df["pickup_community_area"].astype(int) - 1)
    encoded["dropoff_area"] = np.where(df["dropoff_community_area"] > 0, df["dropoff_community_area"], 0).astype(int)
    encoded["holiday"] = (df["is_holiday"].astype(int))

    return encoded

train_cat = encode_categories(train_df)

test_cat = encode_categories(test_df)


## Create TensorFlow Input Dictionaries


In [ ]:
def make_input_dict(
    numeric_array,
    categorical_df
):
    return {
        "numeric": numeric_array,
        "hour": categorical_df["hour"].to_numpy(dtype="int32"),
        "day_of_week": categorical_df["day_of_week"].to_numpy(dtype="int32"),
        "month": categorical_df["month"].to_numpy(dtype="int32"),
        "pickup_area": categorical_df["pickup_area"].to_numpy(dtype="int32"),
        "dropoff_area": categorical_df["dropoff_area"].to_numpy(dtype="int32"),
        "holiday": categorical_df["holiday"].to_numpy(dtype="int32")
    }

X_train = make_input_dict(train_numeric, train_cat)
X_test = make_input_dict(test_numeric, test_cat)

y_train = (train_df["fare"].to_numpy(dtype="float32"))
y_test = (test_df["fare"].to_numpy(dtype="float32"))


# Part V — Build the Deep-Learning Model

## Why Embeddings?

A community-area code is a category, not a numerical magnitude.

For example:

```text
Area 20 is not twice Area 10
```

An embedding lets the network learn a compact representation for each category.

The network can then learn interactions among:

```text
pickup area
× drop-off area
× hour
× demand
× distance
× duration
```

without manually constructing every interaction.


##  Neural-Network Architecture

```text
Continuous features
       +
Categorical embeddings
       ↓
Concatenate
       ↓
Dense 256
       ↓
BatchNorm + ReLU + Dropout
       ↓
Dense 128
       ↓
BatchNorm + ReLU + Dropout
       ↓
Dense 64
       ↓
Predicted fare
```


In [ ]:
def build_model(learning_rate=0.001, dropout=0.10,
                hidden_units=(256, 128, 64), embedding_scale=1.0):

    numeric_input = keras.Input(
        shape=(len(numeric_features),), name="numeric"
    )

    categorical_specs = {
        "hour": (24, max(2, int(6 * embedding_scale))),
        "day_of_week": (7, max(2, int(3 * embedding_scale))),
        "month": (12, max(2, int(4 * embedding_scale))),
        "pickup_area": (77, max(4, int(12 * embedding_scale))),
        "dropoff_area": (78, max(4, int(12 * embedding_scale))),
        "holiday": (2, 2)
    }

    model_inputs = {"numeric": numeric_input}
    encoded_inputs = [numeric_input]

    # Create embedding for each categorical feature
    for name, (vocab_size, embedding_dim) in categorical_specs.items():
        input_layer = keras.Input(shape=(), dtype="int32", name=name)

        embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            name=f"{name}_embedding"
        )(input_layer)

        embedding = layers.Flatten()(embedding)

        model_inputs[name] = input_layer
        encoded_inputs.append(embedding)

    # Combine numerical features and categorical embeddings
    x = layers.Concatenate()(encoded_inputs)

    # Hidden layers
    for units in hidden_units:
        x = layers.Dense(units)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Dropout(dropout)(x)

    # Regression output
    output = layers.Dense(1, name="fare")(x)

    model = keras.Model(inputs=model_inputs, outputs=output)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"]
    )

    return model

##  Build and Inspect the Initial Model

In [ ]:
deep_model = build_model()

deep_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ hour (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_of_week         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ month (InputLayer)  │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pickup_area         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropoff_area        │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ holiday             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_embedding      │ (None, 6)         │        144 │ hour[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_of_week_embedd… │ (None, 3)         │         21 │ day_of_week[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ month_embedding     │ (None, 4)         │         48 │ month[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pickup_area_embedd… │ (None, 12)        │        924 │ pickup_area[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropoff_area_embed… │ (None, 12)        │        936 │ dropoff_area[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ holiday_embedding   │ (None, 2)         │          4 │ holiday[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric             │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 6)         │          0 │ hour_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 3)         │          0 │ day_of_week_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 4)         │          0 │ month_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 12)        │          0 │ pickup_area_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 12)        │          0 │ dropoff_area_emb… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 2)         │          0 │ holiday_embeddin… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 56,606 (221.12 KB)

 Trainable params: 55,710 (217.62 KB)

 Non-trainable params: 896 (3.50 KB)

# Part VI — Train the Initial Model

## Early Stopping

Training stops when validation loss no longer improves, and the best model weights are restored.


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5
)


## Train the Model

15% of the training data is used internally for validation.

The test set remains untouched.


In [ ]:
history = deep_model.fit(
    X_train,
    y_train,
    validation_split=0.15,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=1
)


Epoch 1/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 57ms/step - loss: 229.5632 - mae: 12.4114 - val_loss: 180.1122 - val_mae: 10.4356 - learning_rate: 0.0010
Epoch 2/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 90.9720 - mae: 7.1957 - val_loss: 64.0809 - val_mae: 5.4627 - learning_rate: 0.0010
Epoch 3/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 51.0716 - mae: 4.6657 - val_loss: 46.5382 - val_mae: 4.2892 - learning_rate: 0.0010
Epoch 4/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 43.6208 - mae: 4.0793 - val_loss: 41.0257 - val_mae: 3.8662 - learning_rate: 0.0010
Epoch 5/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 42.9649 - mae: 4.0619 - val_loss: 40.1607 - val_mae: 3.8706 - learning_rate: 0.0010
Epoch 6/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 42.5973 - mae: 4.0508 - val_loss: 39.7242 - val_mae: 3.8274 - learning_rate: 0.0010
Epoch 7/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 42.2319 - mae: 4.0310 - val_loss: 39.6690 - val_mae: 3.8283 - l

## Plot Training and Validation Loss


In [ ]:
history_df = pd.DataFrame(history.history)
history_df["epoch"] = np.arange(1, len(history_df) + 1)

fig = px.line(
    history_df,
    x="epoch",
    y=["loss", "val_loss"],
    title="Deep Learning Training and Validation Loss",
    labels={"value": "MSE Loss", "epoch": "Epoch", "variable": "Series"}
)

fig.update_layout(title=dict(x=0.5, xanchor="center"))
fig.show()

# Part VII — Test Performance

## Predict the Test Set


In [ ]:
deep_pred = deep_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
).ravel()


49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


## R², MAE, and RMSE


In [ ]:
deep_r2 = r2_score(y_test, deep_pred)
deep_mae = mean_absolute_error(y_test, deep_pred)
deep_rmse = np.sqrt(mean_squared_error(y_test, deep_pred))

print(f"Deep Learning R²:   {deep_r2:.4f}")
print(f"Deep Learning MAE:  ${deep_mae:.2f}")
print(f"Deep Learning RMSE: ${deep_rmse:.2f}")

Deep Learning R²:   0.7995
Deep Learning MAE:  $3.76
Deep Learning RMSE: $6.10


## Actual vs. Predicted Fare


In [ ]:
prediction_df = pd.DataFrame({
    "Actual Fare": y_test,
    "Predicted Fare": deep_pred
})

plot_sample = prediction_df.sample(
    n=min(10_000, len(prediction_df)),
    random_state=RANDOM_STATE
)

fig = px.scatter(
    plot_sample,
    x="Actual Fare",
    y="Predicted Fare",
    opacity=0.3,
    title="Deep Learning — Actual vs. Predicted Fare"
)

max_value = max(
    plot_sample["Actual Fare"].max(),
    plot_sample["Predicted Fare"].max()
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_value,
    y1=max_value,
    line=dict(dash="dash")
)

fig.update_layout(title=dict(x=0.5, xanchor="center"))
fig.show()

##  Compare with Notebook 04

Replace these four placeholder R² values with the exact values from your final Notebook 04 run.


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline Linear",
        "Full Linear",
        "Polynomial Ridge",
        "Gradient Boosting",
        "Deep Learning"
    ],
    "R2": [0.73, 0.76, 0.77, 0.78, deep_r2]
})

display(comparison.round(4))

fig = px.bar(
    comparison,
    x="Model",
    y="R2",
    text=comparison["R2"].round(4),
    title="Traditional Models vs. Deep Learning",
    labels={"R2": "Test R²"}
)

fig.update_layout(title=dict(x=0.5, xanchor="center"))
fig.show()

,Model,R2
0,Baseline Linear,0.7300
1,Full Linear,0.7600
2,Polynomial Ridge,0.7700
3,Gradient Boosting,0.7800
4,Deep Learning,0.7995


# Part VIII — Optional Hyperparameter Tuning

The initial model uses reasonable default settings.

Now I test whether tuning can improve performance.

Instead of a full GridSearch, I use **Random Search** because training neural networks is expensive.

The tuning search explores:

- learning rate
- dropout
- embedding size
- network architecture


## Import Keras Tuner

In [ ]:
import keras_tuner as kt


## Define the Tunable Model

In [ ]:
def build_tunable_model(hp):
    learning_rate = hp.Choice("learning_rate", [1e-3, 5e-4, 1e-4])
    dropout = hp.Choice("dropout", [0.0, 0.1, 0.2])
    embedding_scale = hp.Choice("embedding_scale", [0.75, 1.0, 1.5])
    architecture = hp.Choice("architecture", ["small", "medium", "large"])

    if architecture == "small":
        hidden_units = (128, 64)
    elif architecture == "medium":
        hidden_units = (256, 128, 64)
    else:
        hidden_units = (512, 256, 128)

    return build_model(
        learning_rate=learning_rate,
        dropout=dropout,
        hidden_units=hidden_units,
        embedding_scale=embedding_scale
    )

## Create a Tuning Subset

The tuning subset comes only from the training data.

The test set remains untouched.


In [ ]:
TUNING_ROWS = min(200_000, len(train_df))

rng = np.random.default_rng(RANDOM_STATE)

tuning_idx = rng.choice(
    len(train_df),
    size=TUNING_ROWS,
    replace=False
)

X_tune = {
    name: values[tuning_idx]
    for name, values in X_train.items()
}

y_tune = y_train[tuning_idx]

print(f"Tuning rows: {len(y_tune):,}")

Tuning rows: 200,000


## Run Random Search

The first search uses 10 trials to keep Colab runtime manageable.


In [ ]:
tuner = kt.RandomSearch(
    build_tunable_model,
    objective="val_loss",
    max_trials=10,
    executions_per_trial=1,
    overwrite=True,
    directory="/content/keras_tuner",
    project_name="chicago_fare"
)

tuner.search(
    X_tune,
    y_tune,
    validation_split=0.15,
    epochs=50,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        )
    ],
    verbose=1
)


Trial 10 Complete [00h 00m 43s]
val_loss: 48.72789764404297

Best val_loss So Far: 40.25516128540039
Total elapsed time: 00h 06m 10s


##  Best Hyperparameters

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Best learning rate:", best_hp.get("learning_rate"))
print("Best dropout:", best_hp.get("dropout"))
print("Best embedding scale:", best_hp.get("embedding_scale"))
print("Best architecture:", best_hp.get("architecture"))

Best learning rate: 0.0005
Best dropout: 0.1
Best embedding scale: 1.5
Best architecture: medium


## Retrain the Winning Model on the Full Training Set


In [ ]:
tuned_model = tuner.hypermodel.build(
    best_hp
)

tuned_history = tuned_model.fit(
    X_train,
    y_train,
    validation_split=0.15,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5
        )
    ],
    verbose=1
)


Epoch 1/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 343.2211 - mae: 15.2751 - val_loss: 347.5294 - val_mae: 14.8398 - learning_rate: 5.0000e-04
Epoch 2/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 225.9185 - mae: 12.3962 - val_loss: 211.0917 - val_mae: 11.8744 - learning_rate: 5.0000e-04
Epoch 3/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 147.0199 - mae: 9.8075 - val_loss: 113.4035 - val_mae: 8.3845 - learning_rate: 5.0000e-04
Epoch 4/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 97.8955 - mae: 7.5981 - val_loss: 76.3891 - val_mae: 6.5183 - learning_rate: 5.0000e-04
Epoch 5/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 70.4584 - mae: 6.0601 - val_loss: 58.5676 - val_mae: 5.3512 - learning_rate: 5.0000e-04
Epoch 6/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 54.3053 - mae: 4.9566 - val_loss: 47.0808 - val_mae: 4.4661 - learning_rate: 5.0000e-04
Epoch 7/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 47.0863 - mae: 4.3575 - val_loss:

## Evaluate the Tuned Model

In [ ]:
tuned_pred = tuned_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
).ravel()

tuned_r2 = r2_score(y_test, tuned_pred)
tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))

print(f"Tuned Deep Learning R²:   {tuned_r2:.4f}")
print(f"Tuned Deep Learning MAE:  ${tuned_mae:.2f}")
print(f"Tuned Deep Learning RMSE: ${tuned_rmse:.2f}")

49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Tuned Deep Learning R²:   0.7980
Tuned Deep Learning MAE:  $3.77
Tuned Deep Learning RMSE: $6.12


## Initial vs. Tuned Deep Learning


In [ ]:
deep_comparison = pd.DataFrame({
    "Model": ["Initial Deep Learning", "Tuned Deep Learning"],
    "R2": [deep_r2, tuned_r2],
    "MAE": [deep_mae, tuned_mae],
    "RMSE": [deep_rmse, tuned_rmse]
})

display(deep_comparison.round(4))

,Model,R2,MAE,RMSE
0,Initial Deep Learning,0.7995,3.7573,6.0981
1,Tuned Deep Learning,0.7980,3.7734,6.1209


# Part IX — Save Results to Google Drive


In [ ]:
deep_comparison.to_csv(
    REPORTS_DIR / "06_deep_learning_results.csv",
    index=False
)

MODEL_DIR = Path(
    "/content/drive/MyDrive/chicago-rideshare-pricing/models"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

tuned_model.save(
    MODEL_DIR / "fare_deep_learning.keras"
)

print("Metrics saved to:", REPORTS_DIR / "06_deep_learning_results.csv")
print("Model saved to:", MODEL_DIR / "fare_deep_learning.keras")

Metrics saved to: /content/drive/MyDrive/chicago-rideshare-pricing/reports/06_deep_learning_results.csv
Model saved to: /content/drive/MyDrive/chicago-rideshare-pricing/models/fare_deep_learning.keras
